# Positional Embeddings — Hands-On Notebook

Companion code for the **Positional Embeddings** topic of the Learn Mechanistic Interpretability curriculum
([learnmechinterp.com/topics/positional-embeddings](https://learnmechinterp.com/topics/positional-embeddings/)).

Everything here is implemented from scratch in PyTorch first, then checked against real models
(GPT-2 small for learned absolute embeddings, Pythia-70M for RoPE) through TransformerLens.

| § | Notebook section | Article section |
|---|---|---|
| 1 | Permutation equivariance, and what the causal mask leaks | Why Attention Needs Position |
| 2 | GPT-2's learned `W_pos`: geometry, token/position separability | Absolute Position Vectors |
| 3 | Sinusoidal encodings: offsets as exact rotations | Fixed Sinusoidal Encodings |
| 4 | RoPE from scratch, validated against Pythia's internals | Rotary Position Embedding |
| 5 | Relative biases, T5 buckets, ALiBi, softmax coupling | Relative Position Inside Attention |
| 6 | Toy transformers with 6 positional schemes: accuracy & length extrapolation | Comparing the Main Families |
| 7 | Interpretability experiments: offset profiles, position shifts, RoPE key re-positioning | Why Position Changes Interpretability |
| 8 | Exercises | — |

**Hardware target:** MacBook Air M4, 24 GB unified memory.
- Pretrained-model analysis runs on **CPU** (GPT-2 small ≈ 0.5 GB, Pythia-70M ≈ 0.3 GB; both fast on the M4 and numerically deterministic, which matters for the exact-equality checks).
- Toy-model training runs on **MPS** if available.
- Peak memory stays well under ~4 GB. Expected total runtime: roughly 5–15 minutes (first run also downloads ~0.8 GB of weights).

## 0. Setup

Run once in your environment (Python 3.10–3.12 recommended):

```bash
pip install torch numpy matplotlib einops transformer_lens
```

In [ ]:
# %pip install torch numpy matplotlib einops transformer_lens

import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")   # CPU fallback for any op MPS lacks
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import math, time
from functools import partial

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
torch.set_grad_enabled(False)          # re-enabled explicitly for training in §6

TRAIN_DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
ANALYSIS_DEVICE = "cpu"                # deterministic float32 for pretrained-model checks

def offset_profile(pattern, max_off, min_q=0):
    # pattern: [B, h, T, T] -> [h, max_off+1]: mean attention from query i to key i - o,
    # averaged over batch and over query positions i >= max(min_q, o).
    out = []
    for o in range(max_off + 1):
        diag = torch.diagonal(pattern, offset=-o, dim1=-2, dim2=-1)   # entries (q = o + t, k = t)
        out.append(diag[..., max(min_q - o, 0):].mean(dim=(0, -1)))
    return torch.stack(out, dim=-1)

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
print(f"torch {torch.__version__} | train device: {TRAIN_DEVICE} | analysis device: {ANALYSIS_DEVICE}")

## 1. Why attention needs position

### 1.1 Self-attention without position is permutation equivariant

For a single head $f(X) = \mathrm{softmax}(XW_Q (XW_K)^\top/\sqrt{d})\,XW_V$. If $P$ permutes rows, then
$f(PX) = P f(X)$. We check this numerically, then check that a causal mask breaks it.

In [ ]:
def attention(x, W_Q, W_K, W_V, causal=False, bias=None):
    # Single-head attention. x: [n, d]. bias: optional [n, n] added to logits.
    q, k, v = x @ W_Q, x @ W_K, x @ W_V
    scores = q @ k.T / math.sqrt(q.shape[-1])
    if bias is not None:
        scores = scores + bias
    if causal:
        n = x.shape[0]
        future = torch.triu(torch.ones(n, n, dtype=torch.bool), diagonal=1)
        scores = scores.masked_fill(future, float("-inf"))
    return torch.softmax(scores, dim=-1) @ v

d, n = 16, 6
x = torch.randn(n, d)
W_Q, W_K, W_V = (torch.randn(d, d) / math.sqrt(d) for _ in range(3))
perm = torch.randperm(n)

out, out_p = attention(x, W_Q, W_K, W_V), attention(x[perm], W_Q, W_K, W_V)
print("No mask : f(PX) == P f(X)?", torch.allclose(out_p, out[perm], atol=1e-5))

out_c, out_cp = attention(x, W_Q, W_K, W_V, causal=True), attention(x[perm], W_Q, W_K, W_V, causal=True)
print("Causal  : f(PX) == P f(X)?", torch.allclose(out_cp, out_c[perm], atol=1e-5))

### 1.2 `dog bites man` vs `man bites dog`

Same token multiset, opposite meaning. Without a mask and without position, the output at every token is
identical across both orderings (just re-ordered). In particular the representation of `bites` cannot tell who bit whom.

In [ ]:
vocab = {"dog": 0, "bites": 1, "man": 2}
E = torch.randn(3, d)

s1 = E[[vocab["dog"], vocab["bites"], vocab["man"]]]
s2 = E[[vocab["man"], vocab["bites"], vocab["dog"]]]

bites_1 = attention(s1, W_Q, W_K, W_V)[1]
bites_2 = attention(s2, W_Q, W_K, W_V)[1]
print("bidirectional, no position: 'bites' identical?", torch.allclose(bites_1, bites_2, atol=1e-6))

bites_1c = attention(s1, W_Q, W_K, W_V, causal=True)[1]
bites_2c = attention(s2, W_Q, W_K, W_V, causal=True)[1]
print("causal, no position      : 'bites' identical?", torch.allclose(bites_1c, bites_2c, atol=1e-6))

### 1.3 The causal mask leaks *some* positional information

The article notes the mask creates an asymmetry but no reusable index. It's still worth seeing that a mask
alone lets a model compute a position-dependent signal: with uniform attention ($W_Q = 0$) and a value that is
non-zero only at a distinguished first token, the output at position $i$ is exactly $1/(i+1)$.

This is the mechanism behind "NoPE" decoder models learning position implicitly. Keep it in mind for §6.

In [ ]:
n = 20
x = torch.zeros(n, 2)
x[0, 0] = 1.0                    # a BOS-like marker in dimension 0
x[:, 1] = 1.0                    # constant feature everywhere
W_Qz, W_Kz = torch.zeros(2, 2), torch.zeros(2, 2)   # uniform attention
W_Vm = torch.tensor([[1.0, 0.0], [0.0, 0.0]])       # value = marker dimension only

signal = attention(x, W_Qz, W_Kz, W_Vm, causal=True)[:, 0]
plt.figure(figsize=(6, 3))
plt.plot(signal, "o-", label="attention output (marker dim)")
plt.plot(1 / (torch.arange(n) + 1.0), "--", label="1/(i+1)")
plt.xlabel("position i"); plt.legend(); plt.title("Causal mask + uniform attention encodes position")
plt.tight_layout(); plt.show()

## 2. Absolute position vectors: GPT-2 small's learned `W_pos`

$$\mathbf r^0_i = W_E[t_i,:] + \mathbf p_i,\qquad W_P\in\mathbb R^{n_{\text{ctx}}\times d_{\text{model}}}$$

We load GPT-2 with `from_pretrained_no_processing`. The default `from_pretrained` applies weight processing
(e.g. `center_writing_weights`) that mean-centres `W_pos`, which would alter the raw geometry we want to inspect.
Attention patterns are unaffected by that processing, so one unprocessed model serves every GPT-2 experiment.

In [ ]:
from transformer_lens import HookedTransformer

gpt2 = HookedTransformer.from_pretrained_no_processing("gpt2", device=ANALYSIS_DEVICE)
gpt2.eval()

W_pos = gpt2.W_pos.detach().float()   # [1024, 768]
W_E   = gpt2.W_E.detach().float()     # [50257, 768]
print("W_pos", tuple(W_pos.shape), "| W_E", tuple(W_E.shape))
print(f"mean ||p_i|| = {W_pos.norm(dim=-1).mean():.2f} | mean ||e_t|| = {W_E.norm(dim=-1).mean():.2f}")

### 2.1 Norms and similarity structure

Look for: (a) how the norm varies with position — especially position 0; (b) whether nearby positions are
similar (a smooth band around the diagonal) — nothing in the training objective forces this, so smoothness is
an *emergent* property of the learned table.

In [ ]:
norms = W_pos.norm(dim=-1)
Wn = F.normalize(W_pos, dim=-1)
cos_sim = Wn @ Wn.T

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(norms)
axes[0].set_xlabel("position"); axes[0].set_ylabel("||p_i||"); axes[0].set_title("Positional embedding norm")
im = axes[1].imshow(cos_sim, cmap="RdBu_r", vmin=-1, vmax=1)
axes[1].set_title("cosine(p_i, p_j)"); axes[1].set_xlabel("j"); axes[1].set_ylabel("i")
axes[1].grid(False); plt.colorbar(im, ax=axes[1])
plt.tight_layout(); plt.show()

for off in [1, 10, 100, 500]:
    print(f"mean cos(p_i, p_(i+{off:>3})) = {torch.diagonal(cos_sim, off).mean():.3f}")

### 2.2 Low-dimensional structure (PCA)

If the table is smooth, a few principal components should capture most of its variance, and the top
components plotted against position should look like low-frequency curves (often described as helix-like).

In [ ]:
P_c = W_pos - W_pos.mean(0, keepdim=True)
U, S, Vh = torch.linalg.svd(P_c, full_matrices=False)
explained = (S**2) / (S**2).sum()
proj = P_c @ Vh[:6].T          # coordinates on the top-6 PCs

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(explained[:30].cumsum(0), "o-")
axes[0].set_title("Cumulative explained variance"); axes[0].set_xlabel("# PCs")
for c in range(4):
    axes[1].plot(proj[:, c], label=f"PC{c+1}")
axes[1].set_title("Top PCs vs position"); axes[1].set_xlabel("position"); axes[1].legend()
sc = axes[2].scatter(proj[:, 0], proj[:, 1], c=torch.arange(len(proj)), cmap="viridis", s=5)
axes[2].set_title("PC1 vs PC2 (colour = position)"); plt.colorbar(sc, ax=axes[2])
plt.tight_layout(); plt.show()
print(f"Top-2 / top-5 / top-10 PCs explain: {explained[:2].sum():.2%} / {explained[:5].sum():.2%} / {explained[:10].sum():.2%}")

### 2.3 Pause and think, empirically: can the model separate identity from position?

Addition does not preserve labelled slots. Two cheap diagnostics:

1. **Subspace overlap.** Principal angles between the top-$k$ PCA subspace of `W_pos` and of `W_E`.
   Cosines near 0 mean nearly orthogonal subspaces (easy to separate linearly); compare against random subspaces.
2. **Linear read-out from the sum.** From $\mathbf x = \mathbf e_t + \mathbf p_i$ alone, try to recover
   (a) the position with a ridge-regression probe and (b) the token by nearest-neighbour against $W_E$.

In [ ]:
def top_subspace(M, k):
    Mc = M - M.mean(0, keepdim=True)
    _, _, Vh = torch.linalg.svd(Mc, full_matrices=False)
    return Vh[:k].T                           # [d, k] orthonormal basis

def principal_cosines(A, B):
    return torch.linalg.svdvals(A.T @ B)      # cosines of principal angles

k = 20
gen = torch.Generator().manual_seed(0)
tok_idx = torch.randperm(W_E.shape[0], generator=gen)[:10_000]   # subsample W_E for a cheap SVD
B_pos = top_subspace(W_pos, k)
B_tok = top_subspace(W_E[tok_idx], k)
B_rand = torch.linalg.qr(torch.randn(W_pos.shape[1], k, generator=gen))[0]

print("Top-5 principal cosines, W_pos vs W_E   :", principal_cosines(B_pos, B_tok)[:5].numpy().round(3))
print("Top-5 principal cosines, W_pos vs random:", principal_cosines(B_pos, B_rand)[:5].numpy().round(3))

In [ ]:
N = 4000
toks = torch.randint(0, W_E.shape[0], (N,), generator=gen)
poss = torch.randint(0, W_pos.shape[0], (N,), generator=gen)
X = W_E[toks] + W_pos[poss]
tr, te = slice(0, 3000), slice(3000, N)

# (a) Ridge probe: x -> position (closed form)
lam = 10.0
Xb = torch.cat([X, torch.ones(N, 1)], dim=1)
y = poss.float()
w = torch.linalg.solve(Xb[tr].T @ Xb[tr] + lam * torch.eye(Xb.shape[1]), Xb[tr].T @ y[tr])
pred = Xb[te] @ w
r2 = 1 - ((pred - y[te])**2).sum() / ((y[te] - y[te].mean())**2).sum()
print(f"(a) Position probe from e_t + p_i: test R^2 = {r2:.3f}, MAE = {(pred - y[te]).abs().mean():.1f} positions")

# (b) Token recovery by cosine nearest neighbour (batched to cap memory at ~200 MB)
W_En = F.normalize(W_E, dim=-1)
def nn_token_acc(vecs, true, bs=1000):
    hits = 0
    for s in range(0, len(vecs), bs):
        hits += ((F.normalize(vecs[s:s+bs], dim=-1) @ W_En.T).argmax(-1) == true[s:s+bs]).sum().item()
    return hits / len(vecs)

print(f"(b) Token NN accuracy from e_t alone      : {nn_token_acc(W_E[toks[te]], toks[te]):.3f}")
print(f"    Token NN accuracy from e_t + p_i       : {nn_token_acc(X[te], toks[te]):.3f}")
print(f"    ... from e_t + p_i - mean(W_pos)       : {nn_token_acc(X[te] - W_pos.mean(0), toks[te]):.3f}")

**Things to write up:** does the position probe work linearly? How much does adding $\mathbf p_i$ hurt naive token
recovery, and how much of that damage is just the shared mean of `W_pos`? Remember that this is only about the
*initial* residual stream; later layers read $\mathbf e_t+\mathbf p_i$ through learned projections.

## 3. Fixed sinusoidal encodings

$$\text{PE}(i,2k)=\sin(i\,\omega_k),\quad \text{PE}(i,2k+1)=\cos(i\,\omega_k),\quad \omega_k=10000^{-2k/d_{\text{model}}}$$

In [ ]:
def sinusoidal_pe(n_pos, d_model, base=10000.0):
    assert d_model % 2 == 0
    pos = torch.arange(n_pos, dtype=torch.float32)[:, None]
    k = torch.arange(d_model // 2, dtype=torch.float32)
    omega = base ** (-2 * k / d_model)
    ang = pos * omega                          # [n_pos, d_model/2]
    pe = torch.zeros(n_pos, d_model)
    pe[:, 0::2] = torch.sin(ang)
    pe[:, 1::2] = torch.cos(ang)
    return pe, omega

PE, omega = sinusoidal_pe(512, 128)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
im = axes[0].imshow(PE, aspect="auto", cmap="RdBu_r"); axes[0].grid(False)
axes[0].set_title("PE[i, dim]"); axes[0].set_xlabel("dimension"); axes[0].set_ylabel("position")
plt.colorbar(im, ax=axes[0])
for kk in [0, 4, 16, 40]:
    axes[1].plot(PE[:100, 2*kk], label=f"sin, pair k={kk}")
axes[1].set_title("Fast vs slow pairs"); axes[1].set_xlabel("position"); axes[1].legend(fontsize=8)
kk = 1
axes[2].plot(PE[:, 2*kk], PE[:, 2*kk+1], alpha=0.3)
sc = axes[2].scatter(PE[:40, 2*kk], PE[:40, 2*kk+1], c=range(40), cmap="viridis")
axes[2].set_aspect("equal"); axes[2].set_title(f"Pair k={kk} as a phase vector (first 40 positions)")
plt.colorbar(sc, ax=axes[2]); plt.tight_layout(); plt.show()

### 3.1 Offsets are exact linear maps

For each pair, shifting position by $\Delta$ is right-multiplication (row-vector convention, $[\sin, \cos]$ ordering) by
$$R_k(\Delta)=\begin{bmatrix}\cos\Delta\omega_k & -\sin\Delta\omega_k\\ \sin\Delta\omega_k & \cos\Delta\omega_k\end{bmatrix}.$$
We (1) verify this block-diagonal $R(\Delta)$ exactly, (2) *learn* a linear map $M$ with least squares from
pairs $(\text{PE}(i), \text{PE}(i+\Delta))$ and check it recovers $R(\Delta)$ — i.e. one matrix works for every $i$.

In [ ]:
def sinusoidal_shift_matrix(delta, omega):
    dm = 2 * len(omega)
    R = torch.zeros(dm, dm)
    c, s = torch.cos(delta * omega), torch.sin(delta * omega)
    for k in range(len(omega)):
        a, b = 2*k, 2*k + 1                         # (sin, cos) slots
        R[a, a], R[a, b] = c[k], -s[k]
        R[b, a], R[b, b] = s[k],  c[k]
    return R

PE64 = PE.double()
delta = 7
R = sinusoidal_shift_matrix(delta, omega.double())
print("PE(i) R(Δ) == PE(i+Δ) for all i?",
      torch.allclose(PE64[:-delta] @ R, PE64[delta:], atol=1e-10))

M = torch.linalg.lstsq(PE64[:-delta], PE64[delta:]).solution   # least-squares M
print(f"Relative error ||M - R|| / ||R|| = {(M - R).norm() / R.norm():.2e}")
print(f"Fit residual = {(PE64[:-delta] @ M - PE64[delta:]).norm():.2e}")

### 3.2 Dot products depend only on the offset

$\text{PE}(i)\cdot\text{PE}(j)=\sum_k \cos((i-j)\omega_k)$. Plot the similarity from two different anchor positions against the offset.

In [ ]:
G = PE @ PE.T
offs = torch.arange(-100, 101)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
im = axes[0].imshow(G, cmap="RdBu_r"); axes[0].grid(False); axes[0].set_title("PE(i)·PE(j)")
plt.colorbar(im, ax=axes[0])
for anchor in [150, 350]:
    axes[1].plot(offs, G[anchor, anchor + offs], label=f"anchor i={anchor}")
axes[1].plot(offs, torch.cos(offs[:, None].float() * omega[None]).sum(-1), "k:", label="Σ cos(Δω)")
axes[1].set_xlabel("offset j - i"); axes[1].legend(); axes[1].set_title("Same curve from any anchor")
plt.tight_layout(); plt.show()

### 3.3 Defined ≠ trained: what extrapolation actually asks for

For a training length $L$, pair $k$ traverses $L\omega_k/2\pi$ cycles. Fast pairs have seen every phase; slow pairs
have only seen a fraction of their circle, so positions beyond $L$ present phases the model never saw there.

In [ ]:
d_model, L_train = 768, 1024
_, om = sinusoidal_pe(1, d_model)
cycles = L_train * om / (2 * math.pi)
plt.figure(figsize=(7, 3.5))
plt.semilogy(cycles)
plt.axhline(1, color="r", ls="--", label="one full cycle")
plt.xlabel("pair k"); plt.ylabel("cycles seen in training")
plt.title(f"d_model={d_model}, L_train={L_train}"); plt.legend(); plt.tight_layout(); plt.show()
print(f"{(cycles < 1).float().mean():.1%} of pairs never complete a cycle within the training length")

## 4. Rotary Position Embedding (RoPE)

$$\tilde{\mathbf q}_i=\mathbf q_i R_i,\quad \tilde{\mathbf k}_j=\mathbf k_j R_j,\quad
\tilde{\mathbf q}_i\tilde{\mathbf k}_j^\top=\mathbf q_i R_iR_j^\top\mathbf k_j^\top,\ \ R_iR_j^\top \text{ depends only on } i-j.$$

Two layouts exist in the wild and are a classic source of bugs when porting weights:
- **interleaved** (RoFormer, GPT-J style): pairs are $(x_0,x_1), (x_2,x_3),\dots$
- **half-split** (GPT-NeoX / Pythia / HF Llama style): pairs are $(x_m, x_{m+d/2})$

Some models also rotate only the first `rotary_dim` coordinates of each head (Pythia-70M rotates 16 of 64).

In [ ]:
def rope_freqs(rot_dim, base=10000.0):
    return base ** (-torch.arange(0, rot_dim // 2, dtype=torch.float32) * 2 / rot_dim)

def apply_rope(x, positions, base=10000.0, style="half", rotary_dim=None):
    # x: [..., pos, dim] (pos must be the second-to-last axis). positions: 1-D tensor [pos].
    # style: "interleaved" pairs (0,1),(2,3)...; "half" pairs (m, m + rot/2).
    rot = x.shape[-1] if rotary_dim is None else rotary_dim
    x_rot, x_pass = x[..., :rot], x[..., rot:]
    omega = rope_freqs(rot, base).to(x.device)
    ang = positions.to(x.device, torch.float32)[:, None] * omega[None, :]   # [pos, rot/2]
    cos, sin = ang.cos().to(x.dtype), ang.sin().to(x.dtype)
    if style == "interleaved":
        x1, x2 = x_rot[..., 0::2], x_rot[..., 1::2]
        out = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)
    elif style == "half":
        x1, x2 = x_rot[..., : rot // 2], x_rot[..., rot // 2 :]
        out = torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1)
    else:
        raise ValueError(style)
    return torch.cat([out, x_pass], dim=-1)

### 4.1 Unit tests of the three defining properties

1. Norm preservation. 2. Shift invariance of $\tilde q_i\cdot\tilde k_j$. 3. The two layouts are the same operator up to a coordinate permutation.

In [ ]:
dh, T = 64, 50
q = torch.randn(T, dh, dtype=torch.float64)
k = torch.randn(T, dh, dtype=torch.float64)
pos = torch.arange(T)

for style in ["interleaved", "half"]:
    rq = apply_rope(q, pos, style=style)
    print(f"[{style:>11}] norms preserved:", torch.allclose(rq.norm(dim=-1), q.norm(dim=-1)))

# Shift invariance: the same content vectors placed at (i, j) and at (i+s, j+s)
qv, kv = torch.randn(dh, dtype=torch.float64), torch.randn(dh, dtype=torch.float64)
def score(i, j, style="half"):
    qi = apply_rope(qv[None], torch.tensor([i]), style=style)[0]
    kj = apply_rope(kv[None], torch.tensor([j]), style=style)[0]
    return (qi @ kj).item()
print("score(10,4), score(110,104), score(1010,1004):",
      [round(score(10, 4), 8), round(score(110, 104), 8), round(score(1010, 1004), 8)])

# Layout equivalence: half(x) == P^{-1} interleaved(P x) where P maps pair m -> (2m, 2m+1)
perm_idx = torch.stack([torch.arange(dh // 2), torch.arange(dh // 2) + dh // 2], dim=-1).flatten()
inv = torch.argsort(perm_idx)
lhs = apply_rope(q, pos, style="half")
rhs = apply_rope(q[:, perm_idx], pos, style="interleaved")[:, inv]
print("half-split == permuted interleaved:", torch.allclose(lhs, rhs))

In [ ]:
# Score over all (i, j) for fixed content: stripes parallel to the diagonal = relative-only dependence
n = 64
Q = apply_rope(qv.expand(n, dh).clone(), torch.arange(n))
K = apply_rope(kv.expand(n, dh).clone(), torch.arange(n))
S = (Q @ K.T).float()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
im = axes[0].imshow(S, cmap="RdBu_r"); axes[0].grid(False)
axes[0].set_title("RoPE score for fixed (q, k) at positions (i, j)"); axes[0].set_xlabel("j"); axes[0].set_ylabel("i")
plt.colorbar(im, ax=axes[0])

# One-pair picture from the article's interactive: q = k = (1, 0), ω = 0.35
w = 0.35
D = torch.arange(-30, 31).float()
axes[1].plot(D, torch.cos(w * D), "o-", ms=3)
axes[1].set_xlabel("offset Δ = i - j"); axes[1].set_ylabel("dot product")
axes[1].set_title("Single pair, q = k = (1,0): score = cos(ωΔ)")
plt.tight_layout(); plt.show()

### 4.2 Frequencies: wavelengths, the "long-term decay", and the base

With $q=k=\mathbf 1$ the score is $\sum_k 2\cos(\Delta\omega_k)$, whose envelope shrinks with $|\Delta|$ (RoFormer's
decay argument). Changing the base (10k in the original, 500k in Llama 3) stretches all wavelengths.
Note the decay is for *this particular* content; a trained head can choose $q,k$ that peak at any offset.

In [ ]:
rot = 128
D = torch.arange(0, 4096).float()
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for base in [500.0, 10_000.0, 500_000.0]:
    om_b = rope_freqs(rot, base)
    axes[0].semilogy(2 * math.pi / om_b, label=f"base={int(base):,}")
    s = 2 * torch.cos(D[:, None] * om_b[None]).sum(-1) / rot
    axes[1].plot(D, s, lw=0.7, label=f"base={int(base):,}")
axes[0].set_xlabel("pair index"); axes[0].set_ylabel("wavelength (tokens)"); axes[0].legend()
axes[0].set_title("Wavelength per rotary pair")
axes[1].set_xscale("symlog", linthresh=10); axes[1].set_xlabel("offset Δ"); axes[1].set_ylabel("normalised score")
axes[1].set_title("q = k = 1: score vs offset"); axes[1].legend()
plt.tight_layout(); plt.show()

### 4.3 Context extension as a phase-range argument (position interpolation)

Train at $L$, infer at $sL$. Plain RoPE feeds slow pairs phases beyond anything seen in training.
Position interpolation divides positions by $s$, keeping every pair's phase range inside the trained range —
at the cost of squeezing fast pairs, so nearby offsets become harder to distinguish. Both are out-of-distribution
interventions; this plot only shows *which* pairs are pushed off-distribution.

In [ ]:
L, s, rot = 2048, 4, 128
om_r = rope_freqs(rot)
trained_max = torch.clamp(L * om_r, max=2 * math.pi)            # phase coverage seen in training (capped at a full turn)
plain_max  = torch.clamp(s * L * om_r, max=2 * math.pi)
interp_max = torch.clamp((s * L / s) * om_r, max=2 * math.pi)

plt.figure(figsize=(8, 3.5))
plt.plot(trained_max, label=f"trained (L={L})", lw=3, alpha=0.5)
plt.plot(plain_max, "--", label=f"plain extrapolation (L={s*L})")
plt.plot(interp_max, ":", label=f"position interpolation (pos / {s})")
plt.xlabel("pair index"); plt.ylabel("phase coverage (rad, capped at 2π)"); plt.legend()
plt.title("Which pairs see unseen phases?"); plt.tight_layout(); plt.show()
print(f"pairs pushed to unseen phases by plain extrapolation: {(plain_max > trained_max + 1e-6).sum().item()} / {rot//2}")

### 4.4 Validate the implementation against a real RoPE model (Pythia-70M)

TransformerLens exposes both the pre-rotation queries/keys (`attn.hook_q`, `attn.hook_k`) and the rotated ones
(`attn.hook_rot_q`, `attn.hook_rot_k`). If our `apply_rope` reproduces `hook_rot_q` from `hook_q`, every later RoPE
experiment is grounded in the model's actual computation.

In [ ]:
pythia = HookedTransformer.from_pretrained_no_processing("pythia-70m", device=ANALYSIS_DEVICE)
pythia.eval()
pcfg = pythia.cfg
ROPE_STYLE = "interleaved" if getattr(pcfg, "rotary_adjacent_pairs", False) else "half"
print(f"pos type={pcfg.positional_embedding_type} | d_head={pcfg.d_head} | rotary_dim={pcfg.rotary_dim} "
      f"| base={pcfg.rotary_base} | style={ROPE_STYLE} | layers={pcfg.n_layers} heads={pcfg.n_heads}")

toks_p = torch.randint(0, pcfg.d_vocab, (2, 64))
_, pc = pythia.run_with_cache(toks_p, names_filter=lambda nm: nm.startswith("blocks.2.attn.hook_") and
                               nm.split(".")[-1] in ("hook_q", "hook_rot_q", "hook_k", "hook_rot_k"))
positions = torch.arange(toks_p.shape[1])
for which in ["q", "k"]:
    raw = pc[f"blocks.2.attn.hook_{which}"]                     # [batch, pos, head, d_head]
    ref = pc[f"blocks.2.attn.hook_rot_{which}"]
    mine = apply_rope(raw.transpose(1, 2), positions, base=pcfg.rotary_base,
                      style=ROPE_STYLE, rotary_dim=pcfg.rotary_dim).transpose(1, 2)
    err = (mine - ref).abs().max().item()
    print(f"hook_rot_{which}: max |ours - TransformerLens| = {err:.2e}  ->  {'OK' if err < 1e-4 else 'MISMATCH: check style/base'}")

### 4.5 Shift invariance inside a real model: Pythia vs GPT-2, layer 0

At layer 0 the residual stream holds only token information in Pythia (no additive position vector), so its
layer-0 attention scores between the same two tokens must be identical whether they sit at $(i,j)$ or
$(i+s, j+s)$. GPT-2's layer 0 sees $\mathbf e_t+\mathbf p_i$, so the same test should fail.
We prepend $s$ random tokens and compare the overlapping block of `hook_attn_scores`.

In [ ]:
def layer0_shift_test(model, s=37, T=48, seed=1):
    g = torch.Generator().manual_seed(seed)
    base_toks = torch.randint(0, model.cfg.d_vocab, (1, T), generator=g)
    prefix = torch.randint(0, model.cfg.d_vocab, (1, s), generator=g)
    shifted = torch.cat([prefix, base_toks], dim=1)
    name = "blocks.0.attn.hook_attn_scores"
    _, cA = model.run_with_cache(base_toks, names_filter=name)
    _, cB = model.run_with_cache(shifted, names_filter=name)
    A = cA[name][0]                       # [head, T, T]
    B = cB[name][0, :, s:, s:]            # same token pairs, positions shifted by s
    tri = torch.tril(torch.ones(T, T, dtype=torch.bool))
    return (A[:, tri] - B[:, tri]).abs().max().item(), A[:, tri].abs().mean().item()

for name, model in [("Pythia-70M (RoPE)", pythia), ("GPT-2 small (learned abs.)", gpt2)]:
    diff, scale = layer0_shift_test(model)
    print(f"{name:<28} max |score(i,j) - score(i+s,j+s)| = {diff:.2e}   (mean |score| = {scale:.2f})")

### 4.6 A RoPE head is a *family* of QK maps indexed by offset

For one layer-0 head, compute the query of token $a$ and the key of token $b$ from the embeddings, then sweep the
offset $\Delta$. The same content pair gets different scores at different separations — collapsing to a single
token×token QK matrix would discard this.

In [ ]:
blk0 = pythia.blocks[0]
words = [" the", " cat", " dog", ".", " and", " of"]
word_ids = [pythia.to_single_token(w_) for w_ in words]
emb = blk0.ln1(pythia.W_E[word_ids].detach())                    # [n_words, d_model]
Qw = torch.einsum("nd,hdk->nhk", emb, blk0.attn.W_Q.detach()) + blk0.attn.b_Q.detach()
Kw = torch.einsum("nd,hdk->nhk", emb, blk0.attn.W_K.detach()) + blk0.attn.b_K.detach()

def offset_scores(qvec, kvec, max_off=40):
    # qvec, kvec: [d_head]; query at position Δ, key at 0.
    offs = torch.arange(0, max_off + 1)
    qr = apply_rope(qvec.expand(len(offs), -1).clone(), offs, base=pcfg.rotary_base,
                    style=ROPE_STYLE, rotary_dim=pcfg.rotary_dim)
    kr = apply_rope(kvec[None], torch.tensor([0]), base=pcfg.rotary_base,
                    style=ROPE_STYLE, rotary_dim=pcfg.rotary_dim)[0]
    return offs, (qr @ kr) / math.sqrt(pcfg.d_head)

fig, axes = plt.subplots(2, 4, figsize=(16, 6), sharex=True)
for h, ax in enumerate(axes.flat):
    for ai, bi in [(1, 0), (2, 1), (3, 4), (0, 5)]:
        offs, sc_ = offset_scores(Qw[ai, h], Kw[bi, h])
        ax.plot(offs, sc_, label=f"q='{words[ai]}' k='{words[bi]}'")
    ax.set_title(f"L0H{h}"); ax.set_xlabel("offset Δ")
axes[0, 0].legend(fontsize=7); plt.suptitle("Pythia-70M layer 0: QK score vs offset for fixed token pairs")
plt.tight_layout(); plt.show()

## 5. Relative position inside attention

$$s_{ij}=\frac{\mathbf q_i\mathbf k_j^\top}{\sqrt{d_k}}+b_{i-j}$$

### 5.1 T5-style bucketed relative bias

T5 learns one scalar per (head, bucket). Small offsets get their own bucket; larger ones share log-spaced buckets up
to a maximum distance, beyond which everything is clipped into the last bucket.

In [ ]:
def t5_relative_bucket(rel, bidirectional=True, num_buckets=32, max_distance=128):
    # rel = key_pos - query_pos (T5 convention).
    ret = torch.zeros_like(rel)
    n = -rel
    if bidirectional:
        num_buckets //= 2
        ret = ret + (n < 0).long() * num_buckets
        n = n.abs()
    else:
        n = n.clamp(min=0)
    max_exact = num_buckets // 2
    is_small = n < max_exact
    large = max_exact + (
        torch.log(n.clamp(min=1).float() / max_exact) / math.log(max_distance / max_exact) * (num_buckets - max_exact)
    ).long()
    large = large.clamp(max=num_buckets - 1)
    return ret + torch.where(is_small, n, large)

rel = torch.arange(-300, 301)
plt.figure(figsize=(9, 3.5))
plt.plot(rel, t5_relative_bucket(rel, bidirectional=True), label="bidirectional (encoder)")
plt.plot(rel, t5_relative_bucket(rel, bidirectional=False), label="causal (decoder)")
plt.xlabel("key_pos - query_pos"); plt.ylabel("bucket id"); plt.legend()
plt.title("T5 relative position buckets"); plt.tight_layout(); plt.show()

### 5.2 ALiBi

$s^{(h)}_{ij}=\mathbf q_i\mathbf k_j^\top/\sqrt{d_k}-m_h(i-j)$ for $j\le i$, with geometric slopes $m_h = 2^{-8h/H}$.

In [ ]:
def alibi_slopes(n_heads):
    def pow2_slopes(n):
        start = 2 ** (-8.0 / n)
        return [start ** (i + 1) for i in range(n)]
    if math.log2(n_heads).is_integer():
        return torch.tensor(pow2_slopes(n_heads))
    closest = 2 ** math.floor(math.log2(n_heads))       # the paper's recipe for non-power-of-2 head counts
    return torch.tensor(pow2_slopes(closest) + pow2_slopes(2 * closest)[0::2][: n_heads - closest])

def alibi_bias(T, n_heads):
    i = torch.arange(T)
    dist = (i[:, None] - i[None, :]).clamp(min=0).float()   # backward distance
    return -alibi_slopes(n_heads)[:, None, None] * dist     # [heads, T, T]; future masked separately

H, T = 8, 32
Bias = alibi_bias(T, H)
print("slopes:", alibi_slopes(H).numpy().round(4))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
tri = torch.tril(torch.ones(T, T, dtype=torch.bool))
im = axes[0].imshow(Bias[3].masked_fill(~tri, float("nan")), cmap="viridis"); axes[0].grid(False)
axes[0].set_title(f"ALiBi bias, head 3 (m={alibi_slopes(H)[3]:.3f})"); plt.colorbar(im, ax=axes[0])
# With zero content scores, what does each head attend to from the last position?
for h in range(H):
    p = torch.softmax(Bias[h, -1], dim=-1).flip(0)          # index = backward distance
    axes[1].plot(p, label=f"h{h}")
axes[1].set_xlabel("backward distance"); axes[1].set_ylabel("attention prob")
axes[1].set_title("Pure ALiBi prior from the last query"); axes[1].legend(ncol=2, fontsize=7)
plt.tight_layout(); plt.show()

### 5.3 Content and position add in the logits but couple through softmax

Fix a content boost $c$ for the key at offset 1 and use one ALiBi slope. The logit for that key is the same at every
sequence length, but its probability is not, because the denominator sums over all eligible sources.

In [ ]:
m, c = alibi_slopes(8)[5].item(), 2.0
lengths = [2, 4, 8, 16, 32, 64, 128, 512, 2048]
probs = []
for Tq in lengths:
    dist = torch.arange(Tq).float()        # backward distances 0..Tq-1 from the last query
    logits = -m * dist
    logits[1] += c                          # content match at offset 1
    probs.append(torch.softmax(logits, 0)[1].item())
plt.figure(figsize=(6.5, 3.5))
plt.semilogx(lengths, probs, "o-", base=2)
plt.xlabel("number of eligible positions"); plt.ylabel("P(attend to offset 1)")
plt.title(f"Identical logit, different probability (m={m:.3f}, c={c})"); plt.tight_layout(); plt.show()

## 6. Comparing the families on a controlled task

A small decoder-only transformer trained with six positional schemes:
`none` (NoPE), `learned` (absolute table), `sinusoidal`, `rope`, `alibi`, and `relative` (learned scalar bias per
head per clipped offset, Shaw/T5-like).

**Tasks** (pick with `TASK`):
- `"offset"`: predict the token exactly `OFFSET` positions back — a purely *relative* rule.
- `"first"`: predict the first token of the sequence — an *absolute* rule whose target gets farther away as the sequence grows.

We train at length 32 and evaluate at 32/64/128/256. For `learned`, rows beyond 32 exist but were never trained.

**Predict before running:** which schemes solve each task in-distribution, and which extrapolate?

In [ ]:
class ToyAttention(nn.Module):
    def __init__(self, d_model, n_heads, pos_type, max_rel=64):
        super().__init__()
        self.h, self.dh, self.pos_type, self.max_rel = n_heads, d_model // n_heads, pos_type, max_rel
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out = nn.Linear(d_model, d_model, bias=False)
        if pos_type == "alibi":
            self.register_buffer("slopes", alibi_slopes(n_heads))
        if pos_type == "relative":
            self.rel_bias = nn.Parameter(torch.zeros(n_heads, max_rel + 1))
        self.last_pattern = None

    def forward(self, x, keep_pattern=False):
        B, T, D = x.shape
        q, k, v = self.qkv(x).split(D, dim=-1)
        q, k, v = (t.view(B, T, self.h, self.dh).transpose(1, 2) for t in (q, k, v))   # [B, h, T, dh]
        pos = torch.arange(T, device=x.device)
        if self.pos_type == "rope":
            q, k = apply_rope(q, pos, style="half"), apply_rope(k, pos, style="half")
        scores = q @ k.transpose(-1, -2) / math.sqrt(self.dh)
        dist = (pos[:, None] - pos[None, :]).clamp(min=0)                               # backward distance
        if self.pos_type == "alibi":
            scores = scores - self.slopes[:, None, None] * dist.float()
        if self.pos_type == "relative":
            scores = scores + self.rel_bias[:, dist.clamp(max=self.max_rel)]            # [h, T, T]
        future = torch.triu(torch.ones(T, T, dtype=torch.bool, device=x.device), diagonal=1)
        pattern = scores.masked_fill(future, float("-inf")).softmax(-1)
        if keep_pattern:
            self.last_pattern = pattern.detach()
        return self.out((pattern @ v).transpose(1, 2).reshape(B, T, D))

class ToyBlock(nn.Module):
    def __init__(self, d_model, n_heads, pos_type):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.attn = ToyAttention(d_model, n_heads, pos_type)
        self.mlp = nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.GELU(), nn.Linear(4 * d_model, d_model))
    def forward(self, x, keep_pattern=False):
        x = x + self.attn(self.ln1(x), keep_pattern)
        return x + self.mlp(self.ln2(x))

class ToyTransformer(nn.Module):
    def __init__(self, vocab, d_model=128, n_heads=4, n_layers=2, pos_type="rope", max_len=512):
        super().__init__()
        self.pos_type = pos_type
        self.embed = nn.Embedding(vocab, d_model)
        if pos_type == "learned":
            self.pos = nn.Embedding(max_len, d_model)
            nn.init.normal_(self.pos.weight, std=0.02)
        if pos_type == "sinusoidal":
            self.register_buffer("pe", sinusoidal_pe(max_len, d_model)[0])
        self.blocks = nn.ModuleList([ToyBlock(d_model, n_heads, pos_type) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.unembed = nn.Linear(d_model, vocab, bias=False)

    def forward(self, tokens, keep_pattern=False):
        T = tokens.shape[1]
        x = self.embed(tokens)
        if self.pos_type == "learned":
            x = x + self.pos(torch.arange(T, device=tokens.device))
        if self.pos_type == "sinusoidal":
            x = x + self.pe[:T]
        for blk in self.blocks:
            x = blk(x, keep_pattern)
        return self.unembed(self.ln_f(x))

In [ ]:
VOCAB, OFFSET, TASK = 64, 3, "offset"      # try TASK = "first" as a second experiment
TRAIN_LEN, EVAL_LENS = 32, [32, 64, 128, 256]
STEPS, BATCH, LR = 1500, 64, 1e-3          # ~tens of seconds per scheme on an M4

def make_batch(B, T, device, task=TASK):
    x = torch.randint(0, VOCAB, (B, T), device=device)
    y = torch.full_like(x, -100)
    if task == "offset":
        y[:, OFFSET:] = x[:, :-OFFSET]
    elif task == "first":
        y[:, 1:] = x[:, :1]
    return x, y

def evaluate(model, T, n_batches=4, device=TRAIN_DEVICE):
    model.eval(); correct = total = 0
    with torch.no_grad():
        for _ in range(n_batches):
            x, y = make_batch(128, T, device)
            pred = model(x).argmax(-1)
            m_ = y != -100
            correct += (pred[m_] == y[m_]).sum().item(); total += m_.sum().item()
    return correct / total

def train_scheme(pos_type, seed=0, device=TRAIN_DEVICE):
    torch.manual_seed(seed)
    model = ToyTransformer(VOCAB, pos_type=pos_type).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=STEPS, pct_start=0.1)
    losses = []
    with torch.enable_grad():
        model.train()
        for step in range(STEPS):
            x, y = make_batch(BATCH, TRAIN_LEN, device)
            loss = F.cross_entropy(model(x).reshape(-1, VOCAB), y.reshape(-1), ignore_index=-100)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
            if step % 25 == 0:
                losses.append(loss.item())
    return model, losses

SCHEMES = ["none", "learned", "sinusoidal", "rope", "alibi", "relative"]
results, curves, models = {}, {}, {}
for pt in SCHEMES:
    t0 = time.time()
    models[pt], curves[pt] = train_scheme(pt)
    results[pt] = [evaluate(models[pt], L) for L in EVAL_LENS]
    print(f"{pt:>10} | " + " | ".join(f"len {L}: {a:.3f}" for L, a in zip(EVAL_LENS, results[pt]))
          + f" | {time.time() - t0:.0f}s")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for pt in SCHEMES:
    axes[0].plot(np.arange(len(curves[pt])) * 25, curves[pt], label=pt)
    axes[1].plot(EVAL_LENS, results[pt], "o-", label=pt)
axes[0].set_yscale("log"); axes[0].set_xlabel("step"); axes[0].set_ylabel("train loss"); axes[0].legend()
axes[0].set_title(f"Training (task={TASK}, len={TRAIN_LEN})")
axes[1].axvline(TRAIN_LEN, color="k", ls=":", label="train length")
axes[1].axhline(1 / VOCAB, color="gray", ls="--", label="chance")
axes[1].set_xscale("log", base=2); axes[1].set_xlabel("eval length"); axes[1].set_ylabel("accuracy")
axes[1].set_title("Length generalisation"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

### 6.1 Look inside: offset profiles of the toy models

For each head, average attention mass at backward distance $\delta$. A scheme that solved `offset` should contain a
head peaking at $\delta=$`OFFSET` (possibly split across layers, e.g. δ=1 then δ=2). Compare how that profile looks
at the training length and at 4× the training length.

In [ ]:
def plot_toy_profiles(pos_type, lengths=(TRAIN_LEN, 4 * TRAIN_LEN), max_off=10):
    model = models[pos_type]; model.eval()
    fig, axes = plt.subplots(len(model.blocks), len(lengths), figsize=(6 * len(lengths), 3 * len(model.blocks)), squeeze=False)
    for c, T in enumerate(lengths):
        x, _ = make_batch(32, T, TRAIN_DEVICE)
        model(x, keep_pattern=True)
        for l, blk in enumerate(model.blocks):
            prof = offset_profile(blk.attn.last_pattern, max_off, min_q=max_off).cpu()
            im = axes[l, c].imshow(prof, cmap="magma", vmin=0, vmax=1, aspect="auto"); axes[l, c].grid(False)
            axes[l, c].set_title(f"{pos_type} | layer {l} | T={T}")
            axes[l, c].set_xlabel("backward distance"); axes[l, c].set_ylabel("head")
    plt.colorbar(im, ax=axes); plt.show()

for pt in ["rope", "learned", "alibi"]:
    plot_toy_profiles(pt)

## 7. Why position changes interpretability

### 7.1 Offset profiles of GPT-2 and Pythia on random tokens

On uniformly random tokens there is no content structure to exploit, so any sharp offset preference is
positional. We score each head by its mean attention at offset 1 (previous-token score).

In [ ]:
def model_offset_profiles(model, T=128, B=8, max_off=8, seed=0, text_tokens=None):
    g = torch.Generator().manual_seed(seed)
    toks = text_tokens if text_tokens is not None else torch.randint(0, model.cfg.d_vocab, (B, T), generator=g)
    _, cache = model.run_with_cache(toks, names_filter=lambda nm: nm.endswith("attn.hook_pattern"))
    # min_q=16 skips early destination positions, where short contexts inflate every offset
    return torch.stack([offset_profile(cache["pattern", l], max_off, min_q=16)
                        for l in range(model.cfg.n_layers)])        # [layers, heads, max_off+1]

prof_gpt2 = model_offset_profiles(gpt2)
prof_pyth = model_offset_profiles(pythia)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, prof, nm in [(axes[0], prof_gpt2, "GPT-2 small"), (axes[1], prof_pyth, "Pythia-70M")]:
    im = ax.imshow(prof[..., 1], cmap="magma", vmin=0, vmax=1); ax.grid(False)
    ax.set_title(f"{nm}: attention at offset 1 (random tokens)"); ax.set_xlabel("head"); ax.set_ylabel("layer")
    plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

def top_heads(prof, k=5, off=1):
    flat = prof[..., off].flatten()
    idx = flat.topk(k).indices
    return [(int(i // prof.shape[1]), int(i % prof.shape[1]), round(flat[i].item(), 3)) for i in idx]

print("GPT-2 top previous-token heads (layer, head, score):", top_heads(prof_gpt2))
print("Pythia top previous-token heads (layer, head, score):", top_heads(prof_pyth))

### 7.2 Positional rule or content match? Random tokens vs natural text

The article warns that an attention pattern alone cannot distinguish a positional rule from a content match that
usually happens one token back. If a head's offset-1 score is about the same on random tokens and on real text,
that's evidence for a positional rule (still not proof — the next experiments intervene directly).

In [ ]:
TEXT = (
    "The river had been rising for three days before anyone in the village took the warnings seriously. "
    "By the time the council met, water had already reached the steps of the old mill, and the farmers "
    "who lived along the eastern bank were moving their animals to higher ground. The engineer who had "
    "designed the levee twenty years earlier was invited to speak, but she only repeated what she had said "
    "at the time: the structure was built for a smaller river, and nobody had planned for the new dams upstream. "
    "Some people blamed the weather, others blamed the budget, and a few quietly packed their cars."
)

def text_vs_random(model, heads):
    tt = model.to_tokens(TEXT)[:, :129]
    prof_text = model_offset_profiles(model, text_tokens=tt)
    prof_rand = model_offset_profiles(model, T=tt.shape[1])
    for (l, h, _) in heads:
        print(f"  L{l}H{h}: offset-1 score random={prof_rand[l, h, 1]:.3f}  text={prof_text[l, h, 1]:.3f}")

print("GPT-2:");  text_vs_random(gpt2, top_heads(prof_gpt2))
print("Pythia:"); text_vs_random(pythia, top_heads(prof_pyth))

### 7.3 Intervention on absolute position: shift GPT-2's position embeddings

Replace `hook_pos_embed` (positions $0..T-1$) with rows $s..s+T-1$ of `W_pos`. Relative offsets are unchanged, but
every absolute index moves. Does the top previous-token head keep its behaviour? What happens to the LM loss?

In [ ]:
def shift_pos_hook(pos_embed, hook, s):
    T = pos_embed.shape[1]
    return W_pos[s : s + T].to(pos_embed.device)[None].expand_as(pos_embed)

text_toks = gpt2.to_tokens(TEXT)
T_text = text_toks.shape[1]
(l_prev, h_prev, _) = top_heads(prof_gpt2, k=1)[0]
shifts = [0, 16, 64, 256, 512, 1024 - T_text]
losses_s, prevscore_s = [], []
for s in shifts:
    saved = {}
    def save_pattern(p, hook):
        saved["p"] = p.detach()
    loss = gpt2.run_with_hooks(
        text_toks, return_type="loss",
        fwd_hooks=[("hook_pos_embed", partial(shift_pos_hook, s=s)),
                   (f"blocks.{l_prev}.attn.hook_pattern", save_pattern)],
    ).item()
    pat = saved["p"][:, h_prev]
    prevscore_s.append(torch.diagonal(pat, offset=-1, dim1=-2, dim2=-1)[:, 8:].mean().item())
    losses_s.append(loss)
    print(f"shift s={s:>4}: loss={loss:.3f} | L{l_prev}H{h_prev} offset-1 score={prevscore_s[-1]:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].plot(shifts, losses_s, "o-"); axes[0].set_xlabel("position shift s"); axes[0].set_ylabel("LM loss")
axes[1].plot(shifts, prevscore_s, "o-"); axes[1].set_xlabel("position shift s"); axes[1].set_ylabel(f"L{l_prev}H{h_prev} offset-1")
plt.tight_layout(); plt.show()

**A caution when writing this up (from the article's patching discussion):** a shifted table is a coherent
counterfactual only if the model's circuits are mostly relative. Also note that moving from $s=0$ changes position 0,
which in GPT-2 is special (its embedding norm is unusual and the BOS token sits there), so part of any loss change
may come from losing that anchor rather than from absolute-index dependence. A good control: keep row 0 fixed and
shift only positions 1..T-1.

### 7.4 Where to patch RoPE: re-position the keys of a single layer

Standard RoPE enters *after* the query/key projections, so the narrowest positional intervention is on the rotated
keys. We keep the content of every key (`hook_k`) but rotate it as if it sat at position $j+\delta$ instead of $j$,
then overwrite `hook_rot_k`. A head that prefers relative offset 1 should now peak at backward distance $1+\delta$.
Values, residual stream, and queries are untouched.

In [ ]:
(l_rope, h_rope, _) = top_heads(prof_pyth, k=1)[0]
_store = {}

def save_k_hook(k, hook):
    _store["k"] = k
    return k

def reposition_rot_k_hook(rot_k, hook, delta):
    k = _store["k"]                                          # [batch, pos, head, d_head], unrotated
    pos = torch.arange(k.shape[1], device=k.device) + delta
    return apply_rope(k.transpose(1, 2), pos, base=pcfg.rotary_base,
                      style=ROPE_STYLE, rotary_dim=pcfg.rotary_dim).transpose(1, 2)

g = torch.Generator().manual_seed(3)
rtoks = torch.randint(0, pcfg.d_vocab, (8, 128), generator=g)
fig = plt.figure(figsize=(7, 3.8))
for delta in [0, 1, 2, 4]:
    def save_pattern(p, hook):
        _store["pattern"] = p.detach()
    pythia.run_with_hooks(
        rtoks, return_type=None,
        fwd_hooks=[(f"blocks.{l_rope}.attn.hook_k", save_k_hook),
                   (f"blocks.{l_rope}.attn.hook_rot_k", partial(reposition_rot_k_hook, delta=delta)),
                   (f"blocks.{l_rope}.attn.hook_pattern", save_pattern)],
    )
    prof = offset_profile(_store["pattern"], 10, min_q=16)[h_rope]
    plt.plot(prof, "o-", label=f"δ = {delta}")
plt.xlabel("backward distance"); plt.ylabel("mean attention")
plt.title(f"Pythia L{l_rope}H{h_rope}: keys rotated as if at j+δ"); plt.legend()
plt.tight_layout(); plt.show()

Sanity check for your write-up: with $\delta=0$ this hook reproduces the unmodified model exactly (validated in §4.4).
Also note that Pythia rotates only `rotary_dim` of `d_head` coordinates, so part of every query-key score is
position-independent. That's why a peak may shift but not move with full sharpness.

## 8. Exercises

1. **NoPE counting (§1.3, §6).** If `none` solves `TASK="offset"` in-distribution, find the mechanism: probe the
   residual stream after layer 0 for position (linear regression, as in §2.3). Does it look like the $1/(i+1)$ signal?
2. **Controlled GPT-2 shift (§7.3).** Repeat the shift experiment keeping `W_pos[0]` fixed. Separate "loss of the BOS
   anchor" from "absolute-index dependence".
3. **QK decomposition for GPT-2 (§4.6 analogue).** At layer 0, split the attention score of the top previous-token head
   into token-token, token-position, position-token and position-position terms from
   $(\mathbf e + \mathbf p)W_Q\,((\mathbf e + \mathbf p)W_K)^\top$. Which term creates the offset-1 preference?
4. **Toy extrapolation.** Re-run §6 with `TRAIN_LEN=64` and `TASK="first"`. Which schemes break, and why does ALiBi's
   recency prior matter here but not for `"offset"`?
5. **Position interpolation in the toy.** For the `rope` toy model, evaluate at length 128 with positions divided by 4
   (modify `ToyAttention.forward`). Compare with plain extrapolation — does it help or hurt on an exact-offset task?
6. **RoPE frequency ablation.** In Pythia, zero out the rotation for the fastest (or slowest) rotary pairs of the top
   previous-token head via a `hook_rot_q`/`hook_rot_k` hook. Which frequencies carry the offset-1 preference?

## References
- Vaswani et al., *Attention Is All You Need*, NeurIPS 2017.
- Su et al., *RoFormer: Enhanced Transformer with Rotary Position Embedding*, 2021.
- Shaw, Uszkoreit, Vaswani, *Self-Attention with Relative Position Representations*, NAACL 2018.
- Press, Smith, Lewis, *Train Short, Test Long: Attention with Linear Biases*, ICLR 2022.
- Raffel et al., *Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer* (T5), JMLR 2020.
- Chen et al., *Extending Context Window of Large Language Models via Positional Interpolation*, 2023.
- Haviv et al., *Transformer Language Models without Positional Encodings Still Learn Positional Information*, 2022.